# Подготовка данных для системы рекомендаций

**Цель:** Трансформировать сырой дамп вакансий HeadHunter (Kaggle) в датасет, полностью совместимый с Pydantic-схемой FastAPI бэкенда.

**Задачи очистки:**
1. Переименование исходных колонок под целевую схему БД.
2. Применение лимитов длины для строк в полях `title` и `description`.
3. Создание отсутствующих обязательных полей (`requirements`, `conditions`, `currency` и др.).
4. Перевод текстовых диапазонов работы в числовые интервалы `experience_min` и `experience_max`.
5. Разделение смешанного поля `Schedule` на независимые признаки `remote_type` и `time_type`.
6. Преобразование строковых массивов `Keys` в валидные списки тегов (база для будущего Content-Based Filtering).
7. Генерация постоянных `author_id` (UUID) на основе уникальных имен работодателей.

In [1]:
import pandas as pd
import ast
import uuid

In [2]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 200)

## 1. Загрузка исходных данных

In [3]:
raw_df = pd.read_csv('../data/raw/IT_vacancies_full.csv')
raw_df.head(2)

,Ids,Employer,Name,Salary,From,To,Experience,Schedule,Keys,Description,Area,Professional roles,Specializations,Profarea names,Published at
0,49313809,Space307,Golang Developer (Кипр),True,251322.0,NaN,От 3 до 6 лет,Полный день,"['Docker', 'Golang', 'Redis', 'Английский язык', 'Kafka']",Мы в Space307 разрабатываем международную торговую платформу. Каждый день у нас в онлайне 255 тысяч уникальных пользователей из 100+ стран. У нас плоская структура и нет просто исполнителей. Кажды...,Санкт-Петербург,"['Программист, разработчик']","['Программирование, Разработка']","['Информационные технологии, интернет, телеком']",2021-12-02 12:15:37+03:00
1,48813842,Монополия,Е-mail маркетолог,True,60900.0,NaN,От 1 года до 3 лет,Полный день,"['Грамотность', 'Написание текстов', 'Грамотная речь', 'Написание статей']","С 2015 года наш IT блок меняет рынок автотранспортной логистики, создавая инновационные решения. Мы разрабатываем технологичные решения для перевозчиков и отправителей грузов, для водителей, транс...",Санкт-Петербург,['Менеджер по маркетингу и рекламе'],['Маркетинг'],"['Информационные технологии, интернет, телеком']",2021-12-02 10:33:15+03:00


In [4]:
raw_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 48564 entries, 0 to 48563
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Ids                 48564 non-null  int64  
 1   Employer            48564 non-null  str    
 2   Name                48564 non-null  str    
 3   Salary              48564 non-null  bool   
 4   From                15399 non-null  float64
 5   To                  10276 non-null  float64
 6   Experience          48564 non-null  str    
 7   Schedule            48564 non-null  str    
 8   Keys                48564 non-null  str    
 9   Description         48564 non-null  str    
 10  Area                48564 non-null  str    
 11  Professional roles  48564 non-null  str    
 12  Specializations     48564 non-null  str    
 13  Profarea names      48564 non-null  str    
 14  Published at        48564 non-null  str    
dtypes: bool(1), float64(2), int64(1), str(11)
memory usage: 5.2 MB


## 2. Удаляем дубликаты применяем ограничения и создаем отсутствующие поля

так как названий с длинной меньше 5 символов очень мало, просто удаляем их. 

In [5]:
raw_df[raw_df['Name'].str.len() < 5]['Name']

19022     DBA
24087     SRE
25013    SDET
28418     DBA
46679      QA
Name: Name, dtype: str

In [6]:
# Оставляем только нужные колонки и переименовываем их
cols_to_keep = ['Ids', 'Name', 'Employer', 'Description', 'Area', 'Professional roles', 'From', 'To', 'Experience', 'Schedule', 'Keys']
df = raw_df[cols_to_keep].copy()
df.columns = ['vacancy_id','title', 'author_name', 'description', 'city', 'requirements', 'salary_min', 'salary_max', 'exp_raw', 'schedule_raw', 'keys_raw']

# удаляем дубликаты
initial_rows = len(df)
df = df.drop_duplicates(subset=['vacancy_id'])
final_rows = len(df)
duplicates = initial_rows - final_rows
percent = (duplicates / initial_rows) * 100

print(f"Было вакансий: {initial_rows}")
print(f"Стало вакансий: {final_rows}")
print(f"Удалено дубликатов: {duplicates} ({percent:.1f}%)")

# Валидация ограничений длины для бекенда
df = df[df['title'].str.len().between(5, 150)]

df['description'] = df['description'].str[:20000]

# Добавляем обязательные conditions
df['conditions'] = "Условия обсуждаются на собеседовании"
df['metro'] = None
df['currency'] = "RUB"

# Принудительно переводим колонки в тип Int64 (поддерживающий NaN)
df['salary_min'] = df['salary_min'].astype('Int64')
df['salary_max'] = df['salary_max'].astype('Int64')

Было вакансий: 48564
Стало вакансий: 47330
Удалено дубликатов: 1234 (2.5%)


## 3. Feature Engineering и Маппинг признаков

преобразуем `Experience` в `experience_min` и `experience_max`, а также `Schedule` в `remote_type` и `time_type`.
в колонках `Experience` и `Schedule` содержится ограниченный набор категорий, поэтому для скорости и удобства используем словари

In [7]:
df['exp_raw'].unique()

<StringArray>
['От 3 до 6 лет', 'От 1 года до 3 лет', 'Нет опыта', 'Более 6 лет']
Length: 4, dtype: str

In [8]:
experience_mapping = {
    "От 1 года до 3 лет": (1, 3),
    "От 3 до 6 лет": (3, 6),
    "Нет опыта": (0, 0),
    "Более 6 лет": (6, None)
}

df['experience_min'] = df['exp_raw'].map(lambda x: experience_mapping.get(x, (None, None))[0])
df['experience_max'] = df['exp_raw'].map(lambda x: experience_mapping.get(x, (None, None))[1])

# Принудительно переводим колонки в тип Int64 (поддерживающий NaN)
df['experience_min'] = df['experience_min'].astype('Int64')
df['experience_max'] = df['experience_max'].astype('Int64')

print("Пропуски в experience_min:", df['experience_min'].isna().sum())
print("Пропуски в experience_max:", df['experience_max'].isna().sum())

Пропуски в experience_min: 0
Пропуски в experience_max: 1566


In [9]:
df['schedule_raw'].unique()

<StringArray>
[     'Полный день', 'Удаленная работа',   'Сменный график',
    'Гибкий график',   'Вахтовый метод']
Length: 5, dtype: str

In [10]:
remote_mapping = {
    'Полный день': 'OFFICE',
    'Удаленная работа': 'REMOTE',
    'Сменный график': 'OFFICE',
    'Гибкий график': 'ANY',
    'Вахтовый метод': 'OFFICE'
}

time_mapping = {
    'Полный день': 'FULL',
    'Удаленная работа': 'FULL',
    'Сменный график': 'PART',
    'Гибкий график': 'PART',
    'Вахтовый метод': 'FULL'
}


df['remote_type'] = df['schedule_raw'].map(remote_mapping).fillna('OFFICE')
df['time_type'] = df['schedule_raw'].map(time_mapping).fillna('FULL')

In [11]:
def clean_requirements(req_string):
    # Проверяем на пропуски
    if pd.isna(req_string) or not isinstance(req_string, str):
        return ""
    
    try:
        # Преобразуем строку "['Роль 1', 'Роль 2']" в настоящий Python-список
        raw_list = ast.literal_eval(req_string)
        
        if isinstance(raw_list, list):
            # Извлекаем элементы, убираем лишние пробелы и склеиваем в одну строку
            # Если ролей несколько, они будут разделены запятой и пробелом
            return ", ".join([str(role).strip() for role in raw_list if pd.notna(role)])
            
    except (ValueError, SyntaxError):
        # Если распарсить не удалось, возвращаем просто очищенную строку
        return str(req_string).strip()
        
    return ""

# Применяем функцию к колонке
df['requirements'] = df['requirements'].apply(clean_requirements)

# Проверяем результат
print("Как стало:", df['requirements'].iloc[0])

Как стало: Программист, разработчик


In [12]:
def clean_tags(keys_string):
    # очищаем строку от NaN, проверяем, что это строка и не пустая
    if pd.isna(keys_string) or not isinstance(keys_string, str) or keys_string.strip() in ["", "[]"]:
        return []
    # пытаемся распарсить строку как список
    try:
        raw_list = ast.literal_eval(keys_string)
        
        if isinstance(raw_list, list):
            cleaned_list = [str(tag).strip().lower() for tag in raw_list if pd.notna(tag) and str(tag).strip() != ""]
            # удаляем дубликаты, сохраняя порядок
            return ", ".join(dict.fromkeys(cleaned_list))

    except (ValueError, SyntaxError):
        return []
        
    return []

df['tags'] = df['keys_raw'].apply(clean_tags)

print(f"Было (строка): {df['keys_raw'].iloc[0]} (тип: {type(df['keys_raw'].iloc[0])})")
print(f"Стало (список): {df['tags'].iloc[0]} (тип: {type(df['tags'].iloc[0])})")

Было (строка): ['Docker', 'Golang', 'Redis', 'Английский язык', 'Kafka'] (тип: <class 'str'>)
Стало (список): docker, golang, redis, английский язык, kafka (тип: <class 'str'>)


## 4. Генерация системных UUID

генерируем уникальный `uuid4` для каждого уникального работодателя, чтобы в дальнейшем сохранить связи между вакансиями одной компании.

In [13]:
unique_employers = df['author_name'].unique()

employer_to_uuid = {employer: str(uuid.uuid4()) for employer in unique_employers}

df['author_id'] = df['author_name'].map(employer_to_uuid)

## 5. оставляем только нужные колонки и сохраняем в csv

In [14]:
final_columns = [
    'vacancy_id', 'title', 'author_name', 'description', 'city', 
    'salary_min', 'salary_max', 'requirements', 'conditions', 'metro', 
    'currency', 'experience_min', 'experience_max', 'tags', 
    'remote_type', 'time_type', 'author_id'
]
processed_df = df[final_columns]

In [15]:
processed_df.to_csv('../data/processed/cleaned_vacancies.csv', index=False)
print(f"Сохранено {len(processed_df)} вакансий.")
processed_df.head(3)

Сохранено 47325 вакансий.


,vacancy_id,title,author_name,description,city,salary_min,salary_max,requirements,conditions,metro,currency,experience_min,experience_max,tags,remote_type,time_type,author_id
0,49313809,Golang Developer (Кипр),Space307,Мы в Space307 разрабатываем международную торговую платформу. Каждый день у нас в онлайне 255 тысяч уникальных пользователей из 100+ стран. У нас плоская структура и нет просто исполнителей. Кажды...,Санкт-Петербург,251322,<NA>,"Программист, разработчик",Условия обсуждаются на собеседовании,None,RUB,3,6,"docker, golang, redis, английский язык, kafka",OFFICE,FULL,c266cc48-8be0-4b5a-8e4f-03b62a9a456c
1,48813842,Е-mail маркетолог,Монополия,"С 2015 года наш IT блок меняет рынок автотранспортной логистики, создавая инновационные решения. Мы разрабатываем технологичные решения для перевозчиков и отправителей грузов, для водителей, транс...",Санкт-Петербург,60900,<NA>,Менеджер по маркетингу и рекламе,Условия обсуждаются на собеседовании,None,RUB,1,3,"грамотность, написание текстов, грамотная речь, написание статей",OFFICE,FULL,3c6612a4-9edd-4df6-b415-65a8d9ec5af9
2,49413720,Оператор call-центра (удаленно),Eden Springs,Что нужно будет делать: Принимать входящие звонки от существующих клиентов Совершать исходящие звонки по базе существующих клиентов; Обрабатывать обращения от существующих клиентов по электронной...,Санкт-Петербург,<NA>,<NA>,"Оператор call-центра, специалист контактного центра",Условия обсуждаются на собеседовании,None,RUB,1,3,"клиентоориентированность, ориентация на результат, crm, электронная почта, телефонные переговоры, прием и распределение телефонных звонков, консультирование, пользователь пк, входящие звонки, исхо...",REMOTE,FULL,97d3e7aa-10fc-4825-9f38-ef5bb0279eaa
